# Model Rekomendasi Pemesanan Stok — Naive Bayes untuk Laravel (Final)

Notebook ini melatih dan mengevaluasi model menggunakan Python/scikit-learn, kemudian mengekspor seluruh parameter inferensi ke **JSON portabel** agar prediksi dapat dijalankan langsung oleh PHP/Laravel tanpa Python, FastAPI, atau file pickle pada lingkungan produksi.

Struktur project:

```text
project_naive_bayes/
├── data/
│   ├── raw/dataset_toko_barokah.csv
│   └── processed/dataset_toko_barokah_modeling.csv
├── model/
│   ├── naive_bayes_rekomendasi_stok_bundle.pkl          # arsip penelitian
│   └── naive_bayes_rekomendasi_stok_laravel.json       # artifact deployment Laravel
├── notebook/
└── output/
```

Output utama mencakup klasifikasi **Perlu Pesan/Tidak**, probabilitas kelas, jumlah pemesanan, metrik evaluasi, artifact JSON untuk Laravel, serta fixture parity test. File `.pkl` tetap dibuat hanya sebagai arsip/reproduksi di Python dan **tidak digunakan oleh aplikasi Laravel**.

Pipeline model: median imputation → Yeo-Johnson PowerTransformer + standardisasi → Gaussian Naive Bayes → threshold hasil validasi → aturan rekomendasi hybrid berbasis kondisi persediaan.

In [ ]:
from pathlib import Path
from datetime import datetime
import json, math, warnings, hashlib, platform

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm

import sklearn

from sklearn.base import clone
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, average_precision_score, balanced_accuracy_score,
    classification_report, ConfusionMatrixDisplay, f1_score,
    mean_absolute_error, precision_score, PrecisionRecallDisplay,
    recall_score, roc_auc_score,
)
from sklearn.model_selection import GridSearchCV
from sklearn.naive_bayes import GaussianNB
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PowerTransformer

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 100)
pd.set_option('display.float_format', lambda x: f'{x:,.4f}')

## 1. Konfigurasi project dan lokasi file

In [ ]:
IN_COLAB=False
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB=True
except ImportError:
    print('Berjalan di luar Google Colab.')

if IN_COLAB:
    PROJECT_ROOT=Path('/content/drive/MyDrive/project_naive_bayes')
else:
    cwd=Path.cwd()
    PROJECT_ROOT=cwd if (cwd/'data').exists() else (cwd.parent if (cwd.parent/'data').exists() else cwd)

def first_existing(paths, required=True):
    for p in paths:
        if p.exists(): return p
    if required:
        raise FileNotFoundError('File tidak ditemukan. Lokasi diperiksa:\n'+'\n'.join(f'- {p}' for p in paths))
    return None

MODELING_DATA_PATH=first_existing([
    PROJECT_ROOT/'data'/'processed'/'dataset_toko_barokah_modeling.csv',
    PROJECT_ROOT/'data'/'dataset_toko_barokah_modeling.csv',
    PROJECT_ROOT/'dataset_toko_barokah_modeling.csv',
    Path('/mnt/data/dataset_toko_barokah_modeling.csv'),
])
RAW_DATA_PATH=first_existing([
    PROJECT_ROOT/'data'/'raw'/'dataset_toko_barokah.csv',
    PROJECT_ROOT/'data'/'dataset_toko_barokah.csv',
    PROJECT_ROOT/'dataset_toko_barokah.csv',
    Path('/mnt/data/dataset_toko_barokah.csv'),
], required=False)

MODEL_DIR=PROJECT_ROOT/'model'; OUTPUT_DIR=PROJECT_ROOT/'output'; NOTEBOOK_DIR=PROJECT_ROOT/'notebook'
for folder in [MODEL_DIR, OUTPUT_DIR, NOTEBOOK_DIR]: folder.mkdir(parents=True, exist_ok=True)

MIN_RECALL_VALIDATION=0.70
DEFAULT_LEAD_TIME_DAYS=3
DEFAULT_REVIEW_PERIOD_DAYS=7
DEFAULT_SERVICE_LEVEL=0.95
PREDICTION_HORIZON_DAYS=1
DEFAULT_ON_ORDER=0

print('PROJECT_ROOT :', PROJECT_ROOT)
print('MODELING DATA:', MODELING_DATA_PATH)
print('RAW DATA     :', RAW_DATA_PATH)
print('MODEL DIR    :', MODEL_DIR)
print('OUTPUT DIR   :', OUTPUT_DIR)

## 2. Membaca dataset modeling, memvalidasi kolom, dan menentukan target

In [ ]:
df=pd.read_csv(MODELING_DATA_PATH)
required={
    'tanggal','tanggal_target','nama_barang','stok_akhir','barang_keluar',
    'rata_penjualan_7','std_penjualan_7','rata_penjualan_30',
    'std_penjualan_30','cakupan_stok_hari','hari_dalam_minggu',
    'horizon_hari_target','inventory_position','target_stock',
    'target_perlu_pesan_berikutnya','target_jumlah_pesan_berikutnya','split_data'
}
missing=required-set(df.columns)
if missing: raise ValueError(f'Kolom dataset modeling belum lengkap: {sorted(missing)}')

df['tanggal']=pd.to_datetime(df['tanggal'], errors='coerce')
df['tanggal_target']=pd.to_datetime(df['tanggal_target'], errors='coerce')
df['target']=df['target_perlu_pesan_berikutnya'].map({'Tidak':0,'Perlu Pesan':1})
df=(df.drop_duplicates().dropna(subset=['tanggal','tanggal_target','nama_barang','target','split_data'])
      .sort_values(['tanggal_target','nama_barang']).reset_index(drop=True))
df['target']=df['target'].astype(int)

print('Ukuran data:', df.shape)
print('Jumlah barang:', df.nama_barang.nunique())
print('Periode target:', df.tanggal_target.min().date(), 's.d.', df.tanggal_target.max().date())
display(df.head())

## 3. Fitur aman dari leakage dan temporal split

GaussianNB hanya memakai fitur numerik kontinu. Kolom label lama, label saat ini, safety stock, reorder point, target stock, target klasifikasi, target jumlah, dan split tidak dipakai sebagai fitur.

In [ ]:
FEATURES=[
    'stok_akhir','barang_keluar','rata_penjualan_7','std_penjualan_7',
    'rata_penjualan_30','std_penjualan_30','cakupan_stok_hari',
    'hari_dalam_minggu','horizon_hari_target'
]
for c in FEATURES: df[c]=pd.to_numeric(df[c], errors='coerce')

train_df=df[df.split_data=='Train'].copy().reset_index(drop=True)
val_df=df[df.split_data=='Validation'].copy().reset_index(drop=True)
test_df=df[df.split_data=='Test'].copy().reset_index(drop=True)
for name,part in [('Train',train_df),('Validation',val_df),('Test',test_df)]:
    if part.empty or part.target.nunique()<2: raise ValueError(f'Split {name} tidak valid.')
    print(f'{name:10s}: {len(part):6,d} | Perlu Pesan={part.target.mean():.2%} | {part.tanggal_target.min().date()} s.d. {part.tanggal_target.max().date()}')

X_train,y_train=train_df[FEATURES],train_df.target
X_val,y_val=val_df[FEATURES],val_df.target
X_test,y_test=test_df[FEATURES],test_df.target

distribution=df.target.map({0:'Tidak',1:'Perlu Pesan'}).value_counts().rename_axis('kelas').to_frame('jumlah')
distribution['persentase']=distribution.jumlah/len(df)
display(distribution)

## 4. Baseline, pipeline, dan expanding-window cross-validation

In [ ]:
baseline=DummyClassifier(strategy='most_frequent').fit(X_train,y_train)
baseline_pred=baseline.predict(X_test)
baseline_metrics={
    'accuracy':accuracy_score(y_test,baseline_pred),
    'balanced_accuracy':balanced_accuracy_score(y_test,baseline_pred),
    'precision_perlu_pesan':precision_score(y_test,baseline_pred,zero_division=0),
    'recall_perlu_pesan':recall_score(y_test,baseline_pred,zero_division=0),
    'f1_perlu_pesan':f1_score(y_test,baseline_pred,zero_division=0),
}

def expanding_date_splits(dates,target,n_splits=5):
    dates=pd.Series(pd.to_datetime(dates)).reset_index(drop=True)
    target=pd.Series(target).reset_index(drop=True)
    unique=np.array(sorted(dates.unique()))
    boundaries=np.linspace(0,len(unique),min(n_splits,max(2,len(unique)-2))+2,dtype=int)
    splits=[]
    for fold in range(1,len(boundaries)-1):
        tr_dates=unique[:boundaries[fold]]; va_dates=unique[boundaries[fold]:boundaries[fold+1]]
        tr=np.flatnonzero(dates.isin(tr_dates).to_numpy()); va=np.flatnonzero(dates.isin(va_dates).to_numpy())
        if len(tr) and len(va) and target.iloc[tr].nunique()==2 and target.iloc[va].nunique()==2:
            splits.append((tr,va))
    if len(splits)<2: raise ValueError('Fold temporal valid kurang dari dua.')
    return splits

pipeline=Pipeline([
    ('imputer',SimpleImputer(strategy='median')),
    ('power',PowerTransformer(method='yeo-johnson',standardize=True)),
    ('model',GaussianNB()),
])
cv_splits=expanding_date_splits(train_df.tanggal_target,y_train,5)
grid=GridSearchCV(
    pipeline,
    {'model__var_smoothing':np.logspace(-12,-3,10)},
    scoring={'average_precision':'average_precision','f1':'f1','balanced_accuracy':'balanced_accuracy'},
    refit='average_precision',cv=cv_splits,n_jobs=1,return_train_score=False
)
grid.fit(X_train,y_train)
cv_results=(pd.DataFrame(grid.cv_results_)[[
    'param_model__var_smoothing','mean_test_average_precision','mean_test_f1','mean_test_balanced_accuracy'
]].sort_values('mean_test_average_precision',ascending=False).reset_index(drop=True))
print('Parameter terbaik:',grid.best_params_)
print('CV PR-AUC terbaik:',round(grid.best_score_,4))
display(cv_results)

## 5. Memilih threshold pada Validation set

In [ ]:
val_probability=grid.best_estimator_.predict_proba(X_val)[:,1]
rows=[]
for threshold in np.arange(0.01,1.00,0.01):
    pred=(val_probability>=threshold).astype(int)
    rows.append({
        'threshold':round(float(threshold),2),
        'accuracy':accuracy_score(y_val,pred),
        'balanced_accuracy':balanced_accuracy_score(y_val,pred),
        'precision':precision_score(y_val,pred,zero_division=0),
        'recall':recall_score(y_val,pred,zero_division=0),
        'f1':f1_score(y_val,pred,zero_division=0),
    })
threshold_results=pd.DataFrame(rows)
eligible=threshold_results[threshold_results.recall>=MIN_RECALL_VALIDATION]
best_row=(eligible if not eligible.empty else threshold_results).sort_values(
    ['f1','precision','balanced_accuracy'],ascending=False).iloc[0]
BEST_THRESHOLD=float(best_row.threshold)
print('Threshold terbaik:',BEST_THRESHOLD)
display(best_row.to_frame('nilai'))

## 6. Training final dan evaluasi Test set

In [ ]:
train_val=pd.concat([train_df,val_df],ignore_index=True).sort_values(['tanggal_target','nama_barang'])
final_model=clone(grid.best_estimator_).fit(train_val[FEATURES],train_val.target)
test_probability=final_model.predict_proba(X_test)[:,1]
test_prediction=(test_probability>=BEST_THRESHOLD).astype(int)

projected_inventory=np.maximum(0,test_df.inventory_position.to_numpy()-test_df.rata_penjualan_30.to_numpy()*test_df.horizon_hari_target.to_numpy())
test_qty=np.where(test_prediction==1,np.maximum(0,np.ceil(test_df.target_stock.to_numpy()-projected_inventory)),0).astype(int)
actual_qty=test_df.target_jumlah_pesan_berikutnya.to_numpy()
pos_mask=y_test.to_numpy()==1

metrics={
    'accuracy':accuracy_score(y_test,test_prediction),
    'balanced_accuracy':balanced_accuracy_score(y_test,test_prediction),
    'precision_perlu_pesan':precision_score(y_test,test_prediction,zero_division=0),
    'recall_perlu_pesan':recall_score(y_test,test_prediction,zero_division=0),
    'f1_perlu_pesan':f1_score(y_test,test_prediction,zero_division=0),
    'pr_auc':average_precision_score(y_test,test_probability),
    'roc_auc':roc_auc_score(y_test,test_probability),
    'mae_jumlah_pesan_semua_data':mean_absolute_error(actual_qty,test_qty),
    'mae_jumlah_pesan_target_positif':mean_absolute_error(actual_qty[pos_mask],test_qty[pos_mask]) if pos_mask.any() else np.nan,
    'threshold':BEST_THRESHOLD,
    'best_var_smoothing':float(grid.best_params_['model__var_smoothing']),
}
comparison=pd.DataFrame([
    {'model':'Baseline kelas mayoritas',**baseline_metrics},
    {'model':'Gaussian Naive Bayes final',
     'accuracy':metrics['accuracy'],'balanced_accuracy':metrics['balanced_accuracy'],
     'precision_perlu_pesan':metrics['precision_perlu_pesan'],
     'recall_perlu_pesan':metrics['recall_perlu_pesan'],'f1_perlu_pesan':metrics['f1_perlu_pesan']}
]).set_index('model')
display(comparison)
print(classification_report(y_test,test_prediction,target_names=['Tidak','Perlu Pesan'],zero_division=0))
display(pd.DataFrame([metrics]).T.rename(columns={0:'nilai'}))

In [ ]:
fig,ax=plt.subplots(figsize=(6,5))
ConfusionMatrixDisplay.from_predictions(y_test,test_prediction,display_labels=['Tidak','Perlu Pesan'],values_format='d',ax=ax)
ax.set_title('Confusion Matrix — Test Set'); plt.tight_layout()
CONFUSION_PATH=OUTPUT_DIR/'confusion_matrix.png'; fig.savefig(CONFUSION_PATH,dpi=200,bbox_inches='tight'); plt.show()

fig2,ax2=plt.subplots(figsize=(6,5))
PrecisionRecallDisplay.from_predictions(y_test,test_probability,name='GaussianNB',ax=ax2)
ax2.set_title('Precision–Recall Curve — Test Set'); plt.tight_layout()
PR_PATH=OUTPUT_DIR/'precision_recall_curve.png'; fig2.savefig(PR_PATH,dpi=200,bbox_inches='tight'); plt.show()

## 7. Membentuk fitur kondisi terbaru dari dataset raw

In [ ]:
def latest_features_from_raw(raw,lead=3,review=7,service=0.95,on_order=0,horizon=1):
    required={'tanggal','nama_barang','kategori','barang_keluar','stok_akhir'}
    missing=required-set(raw.columns)
    if missing: raise ValueError(f'Kolom raw belum lengkap: {sorted(missing)}')
    data=raw.copy(); data['tanggal']=pd.to_datetime(data.tanggal,errors='coerce')
    for c in ['barang_keluar','stok_akhir']: data[c]=pd.to_numeric(data[c],errors='coerce')
    data=data.dropna(subset=['tanggal','nama_barang','barang_keluar','stok_akhir']).sort_values(['nama_barang','tanggal'])
    z=norm.ppf(service); output=[]
    for product,g in data.groupby('nama_barang'):
        g=g.sort_values('tanggal').reset_index(drop=True)
        if len(g)<8: continue
        latest=g.iloc[-1]; history=g.iloc[:-1].barang_keluar.astype(float)
        h7=history.tail(7); h30=history.tail(30)
        avg7=float(h7.mean()); std7=float(h7.std(ddof=0)); avg30=float(h30.mean()); std30=float(h30.std(ddof=0))
        safety=z*std30*math.sqrt(lead); reorder=avg30*lead+safety; target_stock=avg30*(lead+review)+safety
        inventory=float(latest.stok_akhir)+float(on_order)
        output.append({
            'tanggal':latest.tanggal,'nama_barang':product,'kategori':latest.kategori,
            'stok_akhir':float(latest.stok_akhir),'barang_keluar':float(latest.barang_keluar),
            'rata_penjualan_7':avg7,'std_penjualan_7':std7,
            'rata_penjualan_30':avg30,'std_penjualan_30':std30,
            'cakupan_stok_hari':float(latest.stok_akhir)/avg30 if avg30>0 else np.nan,
            'hari_dalam_minggu':int(latest.tanggal.dayofweek),'horizon_hari_target':int(horizon),
            'inventory_position':inventory,'safety_stock':math.ceil(safety),
            'reorder_point':math.ceil(reorder),'target_stock':math.ceil(target_stock),
            'lead_time_hari':lead,'review_period_hari':review,'service_level':service,
            'barang_dalam_pemesanan':float(on_order),
        })
    return pd.DataFrame(output)

if RAW_DATA_PATH is not None:
    latest_features=latest_features_from_raw(pd.read_csv(RAW_DATA_PATH),DEFAULT_LEAD_TIME_DAYS,DEFAULT_REVIEW_PERIOD_DAYS,DEFAULT_SERVICE_LEVEL,DEFAULT_ON_ORDER,PREDICTION_HORIZON_DAYS)
    print('Rekomendasi memakai kondisi terakhir dataset raw.')
else:
    latest_features=df.sort_values(['nama_barang','tanggal']).groupby('nama_barang',as_index=False).tail(1).copy()
    print('PERINGATAN: raw tidak ditemukan; memakai baris terakhir dataset modeling.')
print('Jumlah barang:',len(latest_features)); display(latest_features.head())

## 8. Fungsi rekomendasi akhir

### Aturan keputusan hybrid

Rekomendasi akhir menjadi `Perlu Pesan` apabila:

```python
inventory_trigger or (
    model_prediction == 1
    and inventory_quantity > 0
)
```

Dengan demikian, kondisi stok yang sudah mencapai `reorder_point` tetap memicu
pemesanan meskipun probabilitas Naive Bayes belum melewati threshold.


In [ ]:
def rekomendasi_pemesanan(nama_barang,stok_saat_ini=None,barang_dalam_pemesanan=None):
    latest=latest_features[latest_features.nama_barang==nama_barang].sort_values('tanggal').tail(1).copy()
    if latest.empty: raise ValueError(f'Barang {nama_barang!r} tidak ditemukan.')
    on_order=float(latest.iloc[0].get('barang_dalam_pemesanan',0) if barang_dalam_pemesanan is None else barang_dalam_pemesanan)
    if stok_saat_ini is not None:
        stok=float(stok_saat_ini); avg=float(latest.iloc[0].rata_penjualan_30)
        latest.loc[:,'stok_akhir']=stok; latest.loc[:,'inventory_position']=stok+on_order
        latest.loc[:,'cakupan_stok_hari']=stok/avg if avg>0 else np.nan
    else: stok=float(latest.iloc[0].stok_akhir)
    prob=float(final_model.predict_proba(latest[FEATURES])[:,1][0]); model_pred=int(prob>=BEST_THRESHOLD)
    avg=float(latest.iloc[0].rata_penjualan_30); horizon=max(1,int(latest.iloc[0].horizon_hari_target))
    projected=max(0,float(latest.iloc[0].inventory_position)-avg*horizon)
    reorder_point=float(latest.iloc[0].reorder_point)
    target_stock=float(latest.iloc[0].target_stock)

    # Aturan hybrid:
    # 1. Pemesanan wajib dilakukan jika projected inventory sudah
    #    mencapai atau berada di bawah reorder point.
    # 2. Model Naive Bayes tetap dapat memberi peringatan lebih awal.
    inventory_trigger=projected<=reorder_point
    inventory_qty=max(0,math.ceil(target_stock-projected))
    final_need=inventory_trigger or (model_pred==1 and inventory_qty>0)
    qty=inventory_qty if final_need else 0

    return pd.DataFrame([{
        'nama_barang':nama_barang,'tanggal_data_terakhir':pd.Timestamp(latest.iloc[0].tanggal).date(),
        'stok_saat_ini':stok,'barang_dalam_pemesanan':on_order,'projected_inventory':round(projected,2),
        'rata_penjualan_harian':round(avg,4),'safety_stock':int(latest.iloc[0].safety_stock),
        'reorder_point':int(latest.iloc[0].reorder_point),'target_stock':int(latest.iloc[0].target_stock),
        'klasifikasi_model':'Perlu Pesan' if model_pred else 'Tidak',
        'probabilitas_perlu_pesan':round(prob,6),'threshold_model':BEST_THRESHOLD,
        'trigger_inventory':'Ya' if inventory_trigger else 'Tidak',
        'rekomendasi_final':'Perlu Pesan' if final_need else 'Tidak',
        'jumlah_stok_harus_dipesan':qty if final_need else 0,
        'accuracy_model_test':metrics['accuracy'],'balanced_accuracy_test':metrics['balanced_accuracy'],
        'precision_perlu_pesan_test':metrics['precision_perlu_pesan'],
        'recall_perlu_pesan_test':metrics['recall_perlu_pesan'],
        'f1_perlu_pesan_test':metrics['f1_perlu_pesan'],'pr_auc_test':metrics['pr_auc'],
    }])

example='Indomie Goreng' if 'Indomie Goreng' in latest_features.nama_barang.values else latest_features.nama_barang.iloc[0]
display(rekomendasi_pemesanan(example).T.rename(columns={0:'hasil'}))

## 9. Rekomendasi seluruh barang dan ekspor artifact Laravel

Notebook mengekspor dua bentuk model:

1. `.pkl` untuk arsip dan reproduksi penelitian menggunakan Python.
2. `.json` sebagai satu-satunya artifact inference yang dipakai aplikasi Laravel.

Artifact JSON memuat urutan fitur, median imputasi, parameter transformasi Yeo-Johnson, parameter standardisasi, prior/mean/variance GaussianNB, threshold klasifikasi, konfigurasi persediaan, metrik, serta checksum. Notebook juga membuat fixture parity test agar implementasi PHP dapat diverifikasi menghasilkan probabilitas dan keputusan yang sama.

In [ ]:
all_recommendations=pd.concat(
    [rekomendasi_pemesanan(p) for p in sorted(latest_features.nama_barang.dropna().unique())],
    ignore_index=True,
)
all_recommendations=all_recommendations.sort_values(
    ['rekomendasi_final','probabilitas_perlu_pesan','jumlah_stok_harus_dipesan'],
    ascending=[True,False,False],
).reset_index(drop=True)

test_output=test_df[[
    'tanggal','tanggal_target','nama_barang','stok_akhir',
    'target_perlu_pesan_berikutnya','target_jumlah_pesan_berikutnya',
]].copy()
test_output['probabilitas_perlu_pesan']=test_probability
test_output['prediksi_model']=np.where(test_prediction==1,'Perlu Pesan','Tidak')
test_output['jumlah_pesan_prediksi']=test_qty

display(all_recommendations.head(20))

# -----------------------------------------------------------------------------
# A. Arsip model Python. File ini tidak diperlukan pada deployment Laravel.
# -----------------------------------------------------------------------------
PYTHON_MODEL_PATH=MODEL_DIR/'naive_bayes_rekomendasi_stok_bundle.pkl'
joblib.dump({
    'model_pipeline':final_model,
    'threshold':BEST_THRESHOLD,
    'features':FEATURES,
    'label_mapping':{0:'Tidak',1:'Perlu Pesan'},
    'metrics':metrics,
    'minimum_recall_validation':MIN_RECALL_VALIDATION,
    'inventory_default':{
        'lead_time_hari':DEFAULT_LEAD_TIME_DAYS,
        'review_period_hari':DEFAULT_REVIEW_PERIOD_DAYS,
        'service_level':DEFAULT_SERVICE_LEVEL,
        'service_level_z':float(norm.ppf(DEFAULT_SERVICE_LEVEL)),
        'prediction_horizon_days':PREDICTION_HORIZON_DAYS,
        'barang_dalam_pemesanan':DEFAULT_ON_ORDER,
    },
    'trained_at':datetime.now().isoformat(),
},PYTHON_MODEL_PATH)

# -----------------------------------------------------------------------------
# B. Ekstraksi seluruh parameter pipeline menjadi artifact JSON portabel.
# -----------------------------------------------------------------------------
def sha256_file(path, chunk_size=1024*1024):
    digest=hashlib.sha256()
    with open(path,'rb') as f:
        while True:
            chunk=f.read(chunk_size)
            if not chunk: break
            digest.update(chunk)
    return digest.hexdigest()


def json_safe(value):
    """Mengubah objek numpy/pandas menjadi tipe JSON standar tanpa NaN/Infinity."""
    if isinstance(value, dict):
        return {str(k):json_safe(v) for k,v in value.items()}
    if isinstance(value, (list,tuple,np.ndarray,pd.Series)):
        return [json_safe(v) for v in list(value)]
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,float)):
        number=float(value)
        return number if math.isfinite(number) else None
    if isinstance(value, (pd.Timestamp,datetime)):
        return value.isoformat()
    if isinstance(value, Path):
        return str(value)
    return value

imputer=final_model.named_steps['imputer']
power=final_model.named_steps['power']
classifier=final_model.named_steps['model']

trained_at=datetime.now().astimezone().isoformat()
artifact={
    'schema_version':'1.0.0',
    'artifact_type':'gaussian_naive_bayes_php_inference',
    'model_name':'naive_bayes_rekomendasi_stok_toko_barokah',
    'model_version':'1.0.0',
    'deployment':{
        'inference_runtime':'PHP 8.3+ / Laravel',
        'python_required_in_production':False,
        'pickle_required_in_production':False,
        'notes':'Training dilakukan di notebook. Laravel hanya membaca artifact JSON ini.',
    },
    'target':{
        'column':'target_perlu_pesan_berikutnya',
        'classes':[0,1],
        'labels':{'0':'Tidak','1':'Perlu Pesan'},
        'positive_class':1,
    },
    'feature_order':FEATURES,
    'preprocessing':{
        'imputer':{
            'strategy':'median',
            'statistics':imputer.statistics_,
        },
        'power_transformer':{
            'method':'yeo-johnson',
            'standardize':True,
            'lambdas':power.lambdas_,
            'scaler_mean':power._scaler.mean_,
            'scaler_scale':power._scaler.scale_,
            'lambda_zero_tolerance':1e-12,
            'lambda_two_tolerance':1e-12,
        },
    },
    'classifier':{
        'type':'GaussianNB',
        'classes':classifier.classes_,
        'class_count':classifier.class_count_,
        'class_prior':classifier.class_prior_,
        'theta':classifier.theta_,
        'variance':classifier.var_,
        'epsilon':classifier.epsilon_,
        'var_smoothing':classifier.var_smoothing,
        'probability_calculation':'stable_softmax_of_joint_log_likelihood',
    },
    'decision':{
        'probability_threshold':BEST_THRESHOLD,
        'comparison_operator':'>=',
        'model_positive_label':'Perlu Pesan',
        'policy':'hybrid_model_or_inventory_trigger',
        'inventory_trigger':'projected_inventory <= reorder_point',
        'projected_inventory':'max(0, inventory_position - rata_penjualan_30 * horizon_hari_target)',
        'order_quantity':'max(0, ceil(target_stock - projected_inventory)) jika rekomendasi akhir positif',
    },
    'inventory_defaults':{
        'lead_time_hari':DEFAULT_LEAD_TIME_DAYS,
        'review_period_hari':DEFAULT_REVIEW_PERIOD_DAYS,
        'service_level':DEFAULT_SERVICE_LEVEL,
        'service_level_z':float(norm.ppf(DEFAULT_SERVICE_LEVEL)),
        'prediction_horizon_days':PREDICTION_HORIZON_DAYS,
        'barang_dalam_pemesanan':DEFAULT_ON_ORDER,
        'standard_deviation_ddof':0,
        'minimum_history_rows':8,
        'weekday_mapping':{'Monday':0,'Tuesday':1,'Wednesday':2,'Thursday':3,'Friday':4,'Saturday':5,'Sunday':6},
    },
    'metrics_test':metrics,
    'training_metadata':{
        'trained_at':trained_at,
        'python_version':platform.python_version(),
        'scikit_learn_version':sklearn.__version__,
        'train_validation_rows':len(train_val),
        'test_rows':len(test_df),
        'training_target_start':train_val.tanggal_target.min(),
        'training_target_end':train_val.tanggal_target.max(),
        'test_target_start':test_df.tanggal_target.min(),
        'test_target_end':test_df.tanggal_target.max(),
        'modeling_dataset_sha256':sha256_file(MODELING_DATA_PATH),
        'raw_dataset_sha256':sha256_file(RAW_DATA_PATH) if RAW_DATA_PATH else None,
    },
}
artifact=json_safe(artifact)

# Checksum dihitung dari payload tanpa field checksum agar dapat diverifikasi Laravel.
canonical_payload=json.dumps(artifact,ensure_ascii=False,sort_keys=True,separators=(',',':')).encode('utf-8')
artifact['artifact_sha256']=hashlib.sha256(canonical_payload).hexdigest()

LARAVEL_MODEL_PATH=MODEL_DIR/'naive_bayes_rekomendasi_stok_laravel.json'
with open(LARAVEL_MODEL_PATH,'w',encoding='utf-8') as f:
    json.dump(artifact,f,ensure_ascii=False,indent=2,allow_nan=False)

# Manifest checksum atas byte file JSON final. Laravel dapat memverifikasi file
# tanpa perlu mereplikasi canonical JSON encoder milik Python.
LARAVEL_MODEL_SHA256_PATH=MODEL_DIR/'naive_bayes_rekomendasi_stok_laravel.json.sha256'
raw_artifact_sha256=sha256_file(LARAVEL_MODEL_PATH)
LARAVEL_MODEL_SHA256_PATH.write_text(
    f'{raw_artifact_sha256}  {LARAVEL_MODEL_PATH.name}\n',
    encoding='utf-8',
)

# -----------------------------------------------------------------------------
# C. Fixture parity model: input, hasil transformasi, JLL, probabilitas, prediksi.
# -----------------------------------------------------------------------------
parity_parts=[]
for class_value in sorted(test_df.target.unique()):
    class_rows=test_df[test_df.target==class_value]
    parity_parts.append(class_rows.sample(n=min(50,len(class_rows)),random_state=42))
parity_source=(pd.concat(parity_parts,ignore_index=True)
                 .sort_values(['target','tanggal_target','nama_barang'])
                 .reset_index(drop=True))

parity_transformed=final_model[:-1].transform(parity_source[FEATURES])
parity_jll=classifier._joint_log_likelihood(parity_transformed)
parity_probability=final_model.predict_proba(parity_source[FEATURES])
parity_prediction=(parity_probability[:,1]>=BEST_THRESHOLD).astype(int)

parity_model=parity_source[['tanggal','tanggal_target','nama_barang','target']].copy()
for idx,feature in enumerate(FEATURES):
    parity_model[f'input__{feature}']=parity_source[feature].to_numpy()
    parity_model[f'transformed__{feature}']=parity_transformed[:,idx]
parity_model['expected_joint_log_likelihood_0']=parity_jll[:,0]
parity_model['expected_joint_log_likelihood_1']=parity_jll[:,1]
parity_model['expected_probability_0']=parity_probability[:,0]
parity_model['expected_probability_1']=parity_probability[:,1]
parity_model['expected_prediction_class']=parity_prediction
parity_model['expected_prediction_label']=np.where(parity_prediction==1,'Perlu Pesan','Tidak')
PARITY_MODEL_PATH=OUTPUT_DIR/'parity_test_model_laravel.csv'
parity_model.to_csv(PARITY_MODEL_PATH,index=False)

# Fixture parity kebijakan rekomendasi untuk kondisi terbaru seluruh barang.
recommendation_inputs=latest_features.copy()
recommendation_inputs['tanggal_data_terakhir']=pd.to_datetime(recommendation_inputs['tanggal']).dt.date
recommendation_parity=recommendation_inputs.merge(
    all_recommendations,
    on=['nama_barang','tanggal_data_terakhir'],
    how='inner',
    suffixes=('_input','_expected'),
)
PARITY_RECOMMENDATION_PATH=OUTPUT_DIR/'parity_test_rekomendasi_laravel.csv'
recommendation_parity.to_csv(PARITY_RECOMMENDATION_PATH,index=False)

# Output penelitian lainnya.
pd.DataFrame([metrics]).to_csv(OUTPUT_DIR/'metrik_model.csv',index=False)
cv_results.to_csv(OUTPUT_DIR/'hasil_cross_validation.csv',index=False)
threshold_results.to_csv(OUTPUT_DIR/'hasil_threshold_validation.csv',index=False)
test_output.to_csv(OUTPUT_DIR/'prediksi_test.csv',index=False)
all_recommendations.to_csv(OUTPUT_DIR/'rekomendasi_pemesanan_semua_barang.csv',index=False)
with open(OUTPUT_DIR/'metadata_model.json','w',encoding='utf-8') as f:
    json.dump({
        'model':'GaussianNB + MedianImputer + YeoJohnson PowerTransformer',
        'target':'target_perlu_pesan_berikutnya',
        'features':FEATURES,
        'threshold':BEST_THRESHOLD,
        'metrics':json_safe(metrics),
        'deployment_artifact':str(LARAVEL_MODEL_PATH),
        'artifact_payload_sha256':artifact['artifact_sha256'],
        'artifact_file_sha256':raw_artifact_sha256,
        'python_pickle_is_deployment_dependency':False,
        'modeling_data':str(MODELING_DATA_PATH),
        'raw_data':str(RAW_DATA_PATH) if RAW_DATA_PATH else None,
    },f,ensure_ascii=False,indent=2,allow_nan=False)

print('EKSPOR SELESAI')
print('Arsip Python       :',PYTHON_MODEL_PATH)
print('Artifact Laravel   :',LARAVEL_MODEL_PATH)
print('Manifest SHA-256   :',LARAVEL_MODEL_SHA256_PATH)
print('Parity model       :',PARITY_MODEL_PATH)
print('Parity rekomendasi :',PARITY_RECOMMENDATION_PATH)
print('Payload SHA-256    :',artifact['artifact_sha256'])
print('File SHA-256       :',raw_artifact_sha256)
print('Output             :',OUTPUT_DIR)
for p in sorted(OUTPUT_DIR.iterdir()): print('-',p.name)

## 10. Verifikasi artifact tanpa objek scikit-learn

Sel berikut membaca kembali JSON deployment dan menjalankan implementasi referensi menggunakan operasi matematika dasar. Verifikasi ini memastikan artifact sudah memuat seluruh parameter yang diperlukan Laravel.

Kriteria keberhasilan:

- transformasi fitur sama dengan pipeline scikit-learn;
- probabilitas kelas sama dalam toleransi numerik;
- seluruh label prediksi identik;
- tidak ada ketergantungan pada `.pkl` dalam proses inferensi referensi.

In [ ]:
with open(LARAVEL_MODEL_PATH,'r',encoding='utf-8') as f:
    deployment_artifact=json.load(f)


def yeo_johnson_scalar(x, lam, zero_tolerance=1e-12, two_tolerance=1e-12):
    """Rumus Yeo-Johnson yang harus diimplementasikan identik di PHP."""
    if x>=0:
        if abs(lam)<=zero_tolerance:
            return math.log1p(x)
        return (math.pow(x+1.0,lam)-1.0)/lam
    if abs(lam-2.0)<=two_tolerance:
        return -math.log1p(-x)
    return -(math.pow(1.0-x,2.0-lam)-1.0)/(2.0-lam)


def portable_transform(rows, artifact):
    features=artifact['feature_order']
    medians=np.asarray(artifact['preprocessing']['imputer']['statistics'],dtype=float)
    config=artifact['preprocessing']['power_transformer']
    lambdas=np.asarray(config['lambdas'],dtype=float)
    means=np.asarray(config['scaler_mean'],dtype=float)
    scales=np.asarray(config['scaler_scale'],dtype=float)
    values=np.asarray(rows[features],dtype=float).copy()
    values=np.where(np.isnan(values),medians,values)
    transformed=np.empty_like(values,dtype=float)
    for row_index in range(values.shape[0]):
        for feature_index in range(values.shape[1]):
            transformed[row_index,feature_index]=yeo_johnson_scalar(
                float(values[row_index,feature_index]),
                float(lambdas[feature_index]),
                float(config['lambda_zero_tolerance']),
                float(config['lambda_two_tolerance']),
            )
    return (transformed-means)/scales


def portable_predict(rows, artifact):
    z=portable_transform(rows,artifact)
    model=artifact['classifier']
    priors=np.asarray(model['class_prior'],dtype=float)
    theta=np.asarray(model['theta'],dtype=float)
    variance=np.asarray(model['variance'],dtype=float)
    jll=[]
    for class_index in range(len(priors)):
        log_prior=math.log(priors[class_index])
        normalization=-0.5*np.sum(np.log(2.0*math.pi*variance[class_index]))
        distance=-0.5*np.sum(((z-theta[class_index])**2)/variance[class_index],axis=1)
        jll.append(log_prior+normalization+distance)
    jll=np.column_stack(jll)
    row_max=np.max(jll,axis=1,keepdims=True)
    exp_shifted=np.exp(jll-row_max)
    probabilities=exp_shifted/exp_shifted.sum(axis=1,keepdims=True)
    positive_class=int(artifact['target']['positive_class'])
    threshold=float(artifact['decision']['probability_threshold'])
    predictions=(probabilities[:,positive_class]>=threshold).astype(int)
    return z,jll,probabilities,predictions

portable_z,portable_jll,portable_probability,portable_prediction=portable_predict(parity_source,deployment_artifact)
reference_z=final_model[:-1].transform(parity_source[FEATURES])
reference_jll=classifier._joint_log_likelihood(reference_z)
reference_probability=final_model.predict_proba(parity_source[FEATURES])
reference_prediction=(reference_probability[:,1]>=BEST_THRESHOLD).astype(int)

verification={
    'max_abs_error_transformation':float(np.max(np.abs(portable_z-reference_z))),
    'max_abs_error_joint_log_likelihood':float(np.max(np.abs(portable_jll-reference_jll))),
    'max_abs_error_probability':float(np.max(np.abs(portable_probability-reference_probability))),
    'prediction_mismatch_count':int(np.sum(portable_prediction!=reference_prediction)),
}

assert verification['max_abs_error_transformation']<=1e-10, verification
assert verification['max_abs_error_joint_log_likelihood']<=1e-9, verification
assert verification['max_abs_error_probability']<=1e-10, verification
assert verification['prediction_mismatch_count']==0, verification

with open(OUTPUT_DIR/'verifikasi_artifact_laravel.json','w',encoding='utf-8') as f:
    json.dump(verification,f,ensure_ascii=False,indent=2,allow_nan=False)

display(pd.DataFrame([verification]).T.rename(columns={0:'nilai'}))
print('VALID: artifact JSON dapat menjalankan inferensi tanpa objek scikit-learn/pickle.')

## File yang dipakai setelah notebook dijalankan

### Wajib disalin ke aplikasi Laravel

- `model/naive_bayes_rekomendasi_stok_laravel.json`
- `model/naive_bayes_rekomendasi_stok_laravel.json.sha256`
- `output/parity_test_model_laravel.csv`
- `output/parity_test_rekomendasi_laravel.csv`
- `output/verifikasi_artifact_laravel.json`

### Output penelitian dan evaluasi

- `output/metrik_model.csv`
- `output/rekomendasi_pemesanan_semua_barang.csv`
- `output/prediksi_test.csv`
- `output/confusion_matrix.png`
- `output/precision_recall_curve.png`
- `output/hasil_cross_validation.csv`
- `output/hasil_threshold_validation.csv`

### Arsip Python, bukan dependency deployment

- `model/naive_bayes_rekomendasi_stok_bundle.pkl`

Pada deployment produksi, aplikasi hanya membutuhkan Laravel/PHP, MySQL, queue worker apabila digunakan, dan artifact JSON. Python, scikit-learn, FastAPI, serta file `.pkl` tidak diperlukan.